# 05 | Integrar CDC, RAND, BLS e BEA

Cada linha será um estado em um ano (2019–2024). Juntamos taxa de homicídios, registros de regras de armas, desemprego e renda. **Não criamos uma taxa futura para previsão.** Execute antes os notebooks 01, 02, 03 e 04.

## Bloco 1: Abrir as fontes processadas

O 01 gera o arquivo CDC; o 02 gera três arquivos RAND; o 03 prepara desemprego; o 04 prepara renda.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.association_data import build_policy_panel
p = ROOT / "data"
cdc = pd.read_csv(p / "raw/cdc_homicides_2019_2024.csv")
contagem = pd.read_csv(p / "processed/rand_law_counts_2019_2024.csv")
ccw = pd.read_csv(p / "processed/rand_ccw_2019_2024.csv")
antecedentes = pd.read_csv(p / "processed/rand_background_2019_2024.csv")
desemprego = pd.read_csv(p / "processed/bls_unemployment_2019_2024.csv")
renda = pd.read_csv(p / "processed/bea_income_2019_2024.csv")
print({"CDC": len(cdc), "RAND contagem": len(contagem), "RAND porte": len(ccw),
       "RAND antecedentes": len(antecedentes), "BLS": len(desemprego), "BEA": len(renda)})

## Bloco 2: Juntar por estado e ano

A contagem geral de registros RAND é separada entre `Restrictive` e `Permissive`. As categorias específicas de porte e antecedentes permanecem na tabela para análises posteriores.

In [ ]:
painel = build_policy_panel(cdc, contagem, ccw, antecedentes, desemprego, renda)
print("Painel integrado:", painel.shape)
display(painel.head())

## Bloco 3: Conferir e salvar

Esperamos 50 estados em cada um dos seis anos, sem valores vazios. A taxa de homicídios representa homicídios totais por 100 mil habitantes.

In [ ]:
assert painel.shape == (300, 15)
assert painel.groupby("year")["state"].nunique().eq(50).all()
assert not painel.duplicated(["state", "year"]).any()
assert not painel.isna().any().any()
destino = p / "processed/us_gun_policy_panel_2019_2024.csv"
painel.to_csv(destino, index=False)
print("Salvo:", destino.relative_to(ROOT))
display(painel.groupby("year")["state"].nunique())